# Assignment 4.1: Model Store - Exercise 

In [1]:
!pip3 install -U sagemaker

In [2]:
import os
import boto3
from sagemaker.core.helper.session_helper import get_execution_role, Session

role = get_execution_role()
sess = Session()

region = sess.boto_region_name
bucket = sess.default_bucket()
prefix = "packageguard-ai-risk-scoring-xgboost-highlevel"

sm_client = sess.sagemaker_client
s3 = boto3.client("s3", region_name=region)

print("SageMaker session type:", type(sess))
print("Region:", region)
print("Default bucket:", bucket)
print("S3 prefix:", prefix)


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


SageMaker session type: <class 'sagemaker.core.helper.session_helper.Session'>
Region: us-east-1
Default bucket: sagemaker-us-east-1-670238798194
S3 prefix: packageguard-ai-risk-scoring-xgboost-highlevel


---
## Data sources

This section performs data collection and feature engineering for the PackageGuard project. Malicious package labels are gathered from public security sources, package metadata is retrieved from PyPI, and domain-specific features are engineered to capture package behavior and quality characteristics. The resulting dataset is cleaned, transformed into a machine-learning-ready format, and split into training, validation, testing, and production datasets for model development and evaluation.

This version uses a PackageGuard AI data pipeline instead of the breast cancer dataset. The pipeline:

1. Pulls malicious package labels from public GitHub sources.
2. Fetches PyPI package metadata from the PyPI JSON API.
3. Engineers package-level risk features.
4. Produces a model-ready CSV where `label = 1` means known malicious/suspicious and `label = 0` means benign baseline.

The model output is a probability-style risk score. It should support triage and early warning only; it should not be treated as conclusive malware evidence.

## Data preparation

The next cell builds the PackageGuard dataset and prepares the same four files needed by the original lab:

* `train_data.csv`
* `validation_data.csv`
* `batch_data.csv`
* `batch_data_noID.csv`


In [4]:
import pandas as pd
import numpy as np

s3 = boto3.client("s3")

%pip install -q requests scikit-learn pyyaml


import ast
import csv
import io
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Iterable, List, Optional

import pandas as pd
import requests
from sklearn.model_selection import train_test_split


GITHUB_API = "https://api.github.com"
PYPI_API = "https://pypi.org/pypi/{package_name}/json"


POPULAR_PYPI_PACKAGES = [
    "requests", "numpy", "pandas", "flask", "django", "scipy",
    "scikit-learn", "pytest", "fastapi", "pydantic", "boto3",
    "urllib3", "cryptography", "pillow", "sqlalchemy", "matplotlib",
    "jupyter", "tensorflow", "torch", "beautifulsoup4",
]


def github_headers() -> Dict[str, str]:
    headers = {"Accept": "application/vnd.github+json"}
    token = os.getenv("GITHUB_TOKEN")
    if token:
        headers["Authorization"] = f"Bearer {token}"
    return headers


def get_github_tree(owner: str, repo: str, branch: str = "main") -> List[Dict]:
    url = f"{GITHUB_API}/repos/{owner}/{repo}/git/trees/{branch}?recursive=1"
    response = requests.get(url, headers=github_headers(), timeout=30)
    response.raise_for_status()
    data = response.json()

    if data.get("truncated"):
        raise RuntimeError(
            f"GitHub tree was truncated for {owner}/{repo}. "
            "Use GITHUB_TOKEN or reduce the source path."
        )

    return data.get("tree", [])


def download_raw_github_file(owner: str, repo: str, branch: str, path: str) -> str:
    url = f"https://raw.githubusercontent.com/{owner}/{repo}/{branch}/{path}"
    response = requests.get(url, headers=github_headers(), timeout=30)
    response.raise_for_status()
    return response.text


def clean_package_name(value: Optional[str]) -> Optional[str]:
    if value is None:
        return None
    name = str(value).strip().lower()
    return name if name else None


def normalize_ecosystem(value: Optional[str]) -> str:
    if value is None:
        return "unknown"

    ecosystem = str(value).strip().lower()

    if ecosystem in {"pypi", "pip", "python"}:
        return "pypi"

    return ecosystem


def normalize_osv_json(text: str, source: str, reference_url: str) -> List[Dict]:
    records = []

    try:
        obj = json.loads(text)
    except Exception:
        return records

    objects = obj if isinstance(obj, list) else [obj]

    for item in objects:
        if not isinstance(item, dict):
            continue

        affected = item.get("affected") or []
        reason = item.get("summary") or item.get("details") or "malicious package report"
        discovered_date = item.get("published") or item.get("modified") or ""

        for affected_item in affected:
            package = affected_item.get("package") or {}
            package_name = clean_package_name(package.get("name"))
            ecosystem = normalize_ecosystem(package.get("ecosystem"))

            if not package_name:
                continue

            records.append(
                {
                    "package_name": package_name,
                    "ecosystem": ecosystem,
                    "label": 1,
                    "source": source,
                    "reason": str(reason)[:500],
                    "discovered_date": discovered_date,
                    "reference_url": reference_url,
                }
            )

    return records


def normalize_generic_json(text: str, source: str, reference_url: str) -> List[Dict]:
    records = []

    try:
        obj = json.loads(text)
    except Exception:
        return records

    objects = obj if isinstance(obj, list) else [obj]

    for item in objects:
        if not isinstance(item, dict):
            continue

        if "affected" in item:
            records.extend(normalize_osv_json(json.dumps(item), source, reference_url))
            continue

        package_name = None
        for key in ["package_name", "package", "name", "project", "artifact"]:
            if key in item:
                package_name = clean_package_name(item.get(key))
                break

        if not package_name:
            continue

        records.append(
            {
                "package_name": package_name,
                "ecosystem": normalize_ecosystem(
                    item.get("ecosystem") or item.get("registry") or item.get("package_type")
                ),
                "label": 1,
                "source": source,
                "reason": str(
                    item.get("reason")
                    or item.get("description")
                    or item.get("summary")
                    or "curated malicious package record"
                )[:500],
                "discovered_date": item.get("date") or item.get("published") or "",
                "reference_url": reference_url,
            }
        )

    return records


def normalize_generic_csv(text: str, source: str, reference_url: str) -> List[Dict]:
    records = []

    try:
        reader = csv.DictReader(io.StringIO(text))
        fieldnames = [field.lower() for field in (reader.fieldnames or [])]
    except Exception:
        return records

    name_candidates = ["package_name", "package", "name", "project", "artifact"]
    ecosystem_candidates = ["ecosystem", "registry", "package_type"]
    reason_candidates = ["reason", "description", "summary", "malware_type"]

    if not any(col in fieldnames for col in name_candidates):
        return records

    def get_value(row: Dict, candidates: Iterable[str]) -> Optional[str]:
        for candidate in candidates:
            for actual_key in row.keys():
                if actual_key.lower() == candidate:
                    return row.get(actual_key)
        return None

    for row in reader:
        package_name = clean_package_name(get_value(row, name_candidates))
        if not package_name:
            continue

        records.append(
            {
                "package_name": package_name,
                "ecosystem": normalize_ecosystem(get_value(row, ecosystem_candidates)),
                "label": 1,
                "source": source,
                "reason": str(get_value(row, reason_candidates) or "curated malicious package record")[:500],
                "discovered_date": "",
                "reference_url": reference_url,
            }
        )

    return records


def pull_malicious_labels_from_github(
    output_csv: str = "data/external/malicious_packages.csv",
    max_files_per_source: int = 3000,
) -> pd.DataFrame:
    """
    Pull malicious package labels from public GitHub sources.

    Output schema:
        package_name, ecosystem, label, source, reason, discovered_date, reference_url
    """

    sources = [
        {
            "source": "openssf",
            "owner": "ossf",
            "repo": "malicious-packages",
            "branch": "main",
            "path_prefix": "osv",
        },
        {
            "source": "datadog",
            "owner": "DataDog",
            "repo": "malicious-software-packages-dataset",
            "branch": "main",
            "path_prefix": "",
        },
    ]

    all_records = []

    for source in sources:
        print(f"Pulling source: {source['source']}")

        try:
            tree = get_github_tree(source["owner"], source["repo"], source["branch"])
        except Exception as exc:
            print(f"Failed to read tree for {source['source']}: {exc}")
            continue

        files_seen = 0

        for item in tree:
            if item.get("type") != "blob":
                continue

            path = item.get("path", "")
            prefix = source["path_prefix"].strip("/")

            if prefix and not path.startswith(prefix + "/"):
                continue

            if not path.lower().endswith((".json", ".csv")):
                continue

            if files_seen >= max_files_per_source:
                break

            raw_url = (
                f"https://raw.githubusercontent.com/"
                f"{source['owner']}/{source['repo']}/{source['branch']}/{path}"
            )

            try:
                text = download_raw_github_file(
                    source["owner"],
                    source["repo"],
                    source["branch"],
                    path,
                )

                if path.lower().endswith(".json"):
                    records = normalize_generic_json(text, source["source"], raw_url)
                else:
                    records = normalize_generic_csv(text, source["source"], raw_url)

                all_records.extend(records)
                files_seen += 1

            except Exception as exc:
                print(f"Skipping {path}: {exc}")

    labels = pd.DataFrame(all_records)

    if labels.empty:
        labels = pd.DataFrame(
            columns=[
                "package_name",
                "ecosystem",
                "label",
                "source",
                "reason",
                "discovered_date",
                "reference_url",
            ]
        )
    else:
        labels["package_name"] = labels["package_name"].astype(str).str.lower().str.strip()
        labels["ecosystem"] = labels["ecosystem"].fillna("unknown").astype(str).str.lower().str.strip()
        labels = labels[labels["package_name"] != ""]

        labels = labels[labels["ecosystem"].isin(["pypi", "unknown"])]

        labels = labels.drop_duplicates(subset=["package_name", "ecosystem", "source"])

    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    labels.to_csv(output_csv, index=False)

    print(f"Saved malicious labels to {output_csv}. Rows: {len(labels)}")
    return labels


def fetch_pypi_metadata(package_name: str, sleep_seconds: float = 0.05) -> Optional[Dict]:
    url = PYPI_API.format(package_name=package_name)

    try:
        response = requests.get(url, timeout=20)
        time.sleep(sleep_seconds)

        if response.status_code == 404:
            return None

        response.raise_for_status()
        raw = response.json()

        info = raw.get("info", {})
        releases = raw.get("releases", {}) or {}
        urls = info.get("project_urls") or {}
        requires_dist = info.get("requires_dist") or []

        upload_dates = []

        for _, files in releases.items():
            for file_info in files:
                upload_time = file_info.get("upload_time_iso_8601") or file_info.get("upload_time")
                if upload_time:
                    try:
                        upload_dates.append(datetime.fromisoformat(upload_time.replace("Z", "+00:00")))
                    except Exception:
                        pass

        first_release_date = min(upload_dates).isoformat() if upload_dates else ""
        latest_release_date = max(upload_dates).isoformat() if upload_dates else ""

        return {
            "package_name": clean_package_name(info.get("name") or package_name),
            "version": info.get("version"),
            "summary": info.get("summary"),
            "description": info.get("description"),
            "author": info.get("author"),
            "author_email": info.get("author_email"),
            "maintainer": info.get("maintainer"),
            "maintainer_email": info.get("maintainer_email"),
            "license": info.get("license"),
            "requires_dist": requires_dist,
            "num_dependencies_raw": len(requires_dist),
            "releases_count": len(releases),
            "first_release_date": first_release_date,
            "latest_release_date": latest_release_date,
            "home_page": info.get("home_page"),
            "project_url_count": len(urls),
        }

    except Exception as exc:
        print(f"Failed PyPI package {package_name}: {exc}")
        return None


def pull_pypi_metadata(
    package_names: Iterable[str],
    output_csv: str = "data/raw/pypi_metadata.csv",
    max_packages: Optional[int] = None,
) -> pd.DataFrame:
    records = []

    package_list = sorted(set([p for p in package_names if p and not str(p).startswith("#")]))

    if max_packages:
        package_list = package_list[:max_packages]

    for idx, package_name in enumerate(package_list, start=1):
        print(f"[{idx}/{len(package_list)}] PyPI metadata: {package_name}")
        metadata = fetch_pypi_metadata(package_name)
        if metadata:
            records.append(metadata)

    df = pd.DataFrame(records)

    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_csv, index=False)

    print(f"Saved PyPI metadata to {output_csv}. Rows: {len(df)}")
    return df


def parse_list_like(value) -> List[str]:
    if value is None or pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            pass

        return [] if value.strip() == "" else [value]

    return []


def days_since(date_value) -> int:
    if pd.isna(date_value) or not str(date_value).strip():
        return -1

    try:
        dt = datetime.fromisoformat(str(date_value).replace("Z", "+00:00"))
        return max((datetime.now(timezone.utc) - dt).days, 0)
    except Exception:
        return -1


def days_between(start_date, end_date) -> int:
    try:
        if pd.isna(start_date) or pd.isna(end_date):
            return -1

        start = datetime.fromisoformat(str(start_date).replace("Z", "+00:00"))
        end = datetime.fromisoformat(str(end_date).replace("Z", "+00:00"))

        return max((end - start).days, 0)

    except Exception:
        return -1


def text_length(value) -> int:
    if pd.isna(value):
        return 0
    return len(str(value))


def similarity_ratio(a: str, b: str) -> float:
    import difflib

    return difflib.SequenceMatcher(None, str(a).lower(), str(b).lower()).ratio()


def closest_popular_package(package_name: str) -> Dict:
    best_name = ""
    best_score = 0.0

    for popular_name in POPULAR_PYPI_PACKAGES:
        score = similarity_ratio(package_name, popular_name)
        if score > best_score:
            best_name = popular_name
            best_score = score

    return {"closest_popular_package": best_name, "name_similarity_to_popular": best_score}


def engineer_package_features(
    metadata_csv: str,
    labels_csv: str,
    output_csv: str = "data/features/packageguard_features.csv",
) -> pd.DataFrame:

    metadata = pd.read_csv(metadata_csv)
    labels = pd.read_csv(labels_csv)

    metadata["package_name"] = metadata["package_name"].astype(str).str.lower().str.strip()
    labels["package_name"] = labels["package_name"].astype(str).str.lower().str.strip()

    malicious_set = set(labels["package_name"])

    df = metadata.copy()
    df["label"] = df["package_name"].apply(lambda name: 1 if name in malicious_set else 0)

    df["summary_length"] = df["summary"].apply(text_length)
    df["description_length"] = df["description"].apply(text_length)
    df["has_summary"] = (df["summary_length"] > 0).astype(int)
    df["has_description"] = (df["description_length"] > 0).astype(int)

    df["has_author"] = df["author"].apply(lambda x: 0 if pd.isna(x) or str(x).strip() == "" else 1)
    df["has_maintainer"] = df["maintainer"].apply(lambda x: 0 if pd.isna(x) or str(x).strip() == "" else 1)
    df["has_license"] = df["license"].apply(lambda x: 0 if pd.isna(x) or str(x).strip() == "" else 1)
    df["has_home_page"] = df["home_page"].apply(lambda x: 0 if pd.isna(x) or str(x).strip() == "" else 1)

    df["dependencies_list"] = df["requires_dist"].apply(parse_list_like)
    df["num_dependencies"] = df["dependencies_list"].apply(len)

    df["releases_count"] = pd.to_numeric(df["releases_count"], errors="coerce").fillna(0)
    df["project_url_count"] = pd.to_numeric(df["project_url_count"], errors="coerce").fillna(0)

    df["package_age_days"] = df["first_release_date"].apply(days_since)
    df["days_since_latest_release"] = df["latest_release_date"].apply(days_since)

    df["release_span_days"] = df.apply(
        lambda row: days_between(row["first_release_date"], row["latest_release_date"]),
        axis=1,
    )

    df["release_frequency"] = df.apply(
        lambda row: row["releases_count"] / max(row["release_span_days"], 1)
        if row["release_span_days"] >= 0
        else 0,
        axis=1,
    )

    typo_features = df["package_name"].apply(closest_popular_package).apply(pd.Series)
    df = pd.concat([df, typo_features], axis=1)

    df["possible_typosquat"] = (
        (df["name_similarity_to_popular"] >= 0.80)
        & (df["package_name"] != df["closest_popular_package"])
    ).astype(int)

    df["is_new_package"] = (
        (df["package_age_days"] >= 0) & (df["package_age_days"] < 90)
    ).astype(int)

    df["many_dependencies"] = (df["num_dependencies"] >= 10).astype(int)

    df["low_metadata_quality"] = (
        (df["has_description"] == 0)
        | (df["has_license"] == 0)
        | (df["has_home_page"] == 0)
    ).astype(int)

    df["metadata_quality_score"] = (
        df["has_summary"]
        + df["has_description"]
        + df["has_author"]
        + df["has_maintainer"]
        + df["has_license"]
        + df["has_home_page"]
    )

    feature_columns = [
        "package_name",
        "label",
        "summary_length",
        "description_length",
        "has_summary",
        "has_description",
        "has_author",
        "has_maintainer",
        "has_license",
        "has_home_page",
        "num_dependencies",
        "releases_count",
        "project_url_count",
        "package_age_days",
        "days_since_latest_release",
        "release_span_days",
        "release_frequency",
        "closest_popular_package",
        "name_similarity_to_popular",
        "possible_typosquat",
        "is_new_package",
        "many_dependencies",
        "low_metadata_quality",
        "metadata_quality_score",
    ]

    final_df = df[feature_columns].copy()

    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    final_df.to_csv(output_csv, index=False)

    print(f"Saved engineered features to {output_csv}. Rows: {len(final_df)}")
    print("Label distribution:")
    print(final_df["label"].value_counts(dropna=False))

    return final_df


def split_dataset(
    features_csv: str,
    output_dir: str = "data/splits",
    random_state: int = 42,
) -> Dict[str, pd.DataFrame]:
    """
    Split into:
    - train: 40%
    - validation: 10%
    - test: 10%
    - production: 40%
    """

    df = pd.read_csv(features_csv)

    stratify = df["label"] if df["label"].nunique() > 1 and df["label"].value_counts().min() >= 2 else None

    train_df, remaining_df = train_test_split(
        df,
        train_size=0.40,
        random_state=random_state,
        stratify=stratify,
    )

    stratify_remaining = (
        remaining_df["label"]
        if remaining_df["label"].nunique() > 1 and remaining_df["label"].value_counts().min() >= 2
        else None
    )

    validation_df, remaining_df = train_test_split(
        remaining_df,
        train_size=1 / 6,
        random_state=random_state,
        stratify=stratify_remaining,
    )

    stratify_remaining = (
        remaining_df["label"]
        if remaining_df["label"].nunique() > 1 and remaining_df["label"].value_counts().min() >= 2
        else None
    )

    test_df, production_df = train_test_split(
        remaining_df,
        train_size=0.20,
        random_state=random_state,
        stratify=stratify_remaining,
    )

    Path(output_dir).mkdir(parents=True, exist_ok=True)

    train_df.to_csv(f"{output_dir}/train.csv", index=False)
    validation_df.to_csv(f"{output_dir}/validation.csv", index=False)
    test_df.to_csv(f"{output_dir}/test.csv", index=False)
    production_df.to_csv(f"{output_dir}/production.csv", index=False)

    print("Saved splits:")
    print(
        {
            "train": len(train_df),
            "validation": len(validation_df),
            "test": len(test_df),
            "production": len(production_df),
        }
    )

    return {
        "train": train_df,
        "validation": validation_df,
        "test": test_df,
        "production": production_df,
    }


def build_packageguard_dataset(
    max_malicious_packages: int = 500,
    benign_packages: Optional[List[str]] = None,
) -> pd.DataFrame:
    """
    End-to-end function.

    Pulls labels, fetches PyPI metadata, engineers features,
    saves final CSVs, and creates train/validation/test/production splits.
    """

    if benign_packages is None:
        benign_packages = POPULAR_PYPI_PACKAGES

    labels_df = pull_malicious_labels_from_github(
        output_csv="data/external/malicious_packages.csv",
        max_files_per_source=3000,
    )

    malicious_packages = labels_df["package_name"].dropna().astype(str).head(max_malicious_packages).tolist()

    all_packages = sorted(set(benign_packages + malicious_packages))

    metadata_df = pull_pypi_metadata(
        package_names=all_packages,
        output_csv="data/raw/pypi_metadata.csv",
        max_packages=None,
    )

    features_df = engineer_package_features(
        metadata_csv="data/raw/pypi_metadata.csv",
        labels_csv="data/external/malicious_packages.csv",
        output_csv="data/features/packageguard_features.csv",
    )

    split_dataset(
        features_csv="data/features/packageguard_features.csv",
        output_dir="data/splits",
    )

    return features_df

# Build the dataset. Increase max_malicious_packages for a larger run if needed.
data = build_packageguard_dataset(max_malicious_packages=500)

data = data.dropna(subset=["label"]).copy()
data["label"] = data["label"].astype(int)

id_column = "package_name"
target_column = "label"
non_model_columns = [id_column, target_column, "closest_popular_package"]
model_feature_columns = [
    col for col in data.columns
    if col not in non_model_columns and pd.api.types.is_numeric_dtype(data[col])
]

for col in model_feature_columns:
    data[col] = pd.to_numeric(data[col], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)

data = data[[id_column, target_column] + model_feature_columns].copy()

print("Dataset shape:", data.shape)
print("Model feature count:", len(model_feature_columns))
print("Label distribution:")
print(data[target_column].value_counts(dropna=False))
data.head(8)


Note: you may need to restart the kernel to use updated packages.


Pulling source: openssf


Failed to read tree for openssf: GitHub tree was truncated for ossf/malicious-packages. Use GITHUB_TOKEN or reduce the source path.
Pulling source: datadog


Failed to read tree for datadog: GitHub tree was truncated for DataDog/malicious-software-packages-dataset. Use GITHUB_TOKEN or reduce the source path.
Saved malicious labels to data/external/malicious_packages.csv. Rows: 0
[1/20] PyPI metadata: beautifulsoup4
[2/20] PyPI metadata: boto3
[3/20] PyPI metadata: cryptography


[4/20] PyPI metadata: django
[5/20] PyPI metadata: fastapi
[6/20] PyPI metadata: flask


...

[12/20] PyPI metadata: pydantic
[13/20] PyPI metadata: pytest
[14/20] PyPI metadata: requests


[15/20] PyPI metadata: scikit-learn
[16/20] PyPI metadata: scipy
[17/20] PyPI metadata: sqlalchemy


[18/20] PyPI metadata: tensorflow
[19/20] PyPI metadata: torch
[20/20] PyPI metadata: urllib3


Saved PyPI metadata to data/raw/pypi_metadata.csv. Rows: 20
Saved engineered features to data/features/packageguard_features.csv. Rows: 20
Label distribution:
label
0    20
Name: count, dtype: int64
Saved splits:
{'train': 8, 'validation': 2, 'test': 2, 'production': 8}
Dataset shape: (20, 23)
Model feature count: 21
Label distribution:
label
0    20
Name: count, dtype: int64


,package_name,label,summary_length,description_length,has_summary,has_description,has_author,has_maintainer,has_license,has_home_page,...,package_age_days,days_since_latest_release,release_span_days,release_frequency,name_similarity_to_popular,possible_typosquat,is_new_package,many_dependencies,low_metadata_quality,metadata_quality_score
0,beautifulsoup4,0,23,2429,1,1,0,0,1,0,...,4623,181,4442,0.011932,1.0,0,0,0,1,3
1,boto3,0,22,5368,1,1,1,0,1,1,...,4218,1,4216,0.484345,1.0,0,0,0,0,5
2,cryptography,0,99,2252,1,1,0,0,0,0,...,4525,26,4498,0.034460,1.0,0,0,0,1,2
3,django,0,96,2017,1,1,0,0,0,0,...,5857,10,5846,0.073897,1.0,0,0,0,1,2
4,fastapi,0,86,23541,1,1,0,0,0,0,...,2730,7,2723,0.107235,1.0,0,0,1,1,2
5,flask,0,57,1640,1,1,0,0,0,0,...,5888,101,5787,0.011059,1.0,0,0,0,1,2
6,jupyter,0,66,1010,1,1,1,0,1,1,...,3945,638,3306,0.001210,1.0,0,0,0,0,5
7,matplotlib,0,23,3741,1,1,1,0,1,0,...,7446,18,7428,0.018848,1.0,0,0,1,1,4


In [5]:
data[target_column] = data[target_column].astype(int)
data.sample(min(8, len(data)), random_state=42)


,package_name,label,summary_length,description_length,has_summary,has_description,has_author,has_maintainer,has_license,has_home_page,...,package_age_days,days_since_latest_release,release_span_days,release_frequency,name_similarity_to_popular,possible_typosquat,is_new_package,many_dependencies,low_metadata_quality,metadata_quality_score
0,beautifulsoup4,0,23,2429,1,1,0,0,1,0,...,4623,181,4442,0.011932,1.0,0,0,0,1,3
17,tensorflow,0,69,854,1,1,1,0,1,1,...,3449,85,3363,0.041035,1.0,0,0,1,0,5
15,scipy,0,57,3720,1,1,0,0,1,0,...,5786,5,5780,0.018166,1.0,0,0,1,1,3
1,boto3,0,22,5368,1,1,1,0,1,1,...,4218,1,4216,0.484345,1.0,0,0,0,0,5
8,numpy,0,49,4199,1,1,1,0,0,0,...,7120,12,7107,0.020543,1.0,0,0,0,1,3
5,flask,0,57,1640,1,1,0,0,0,0,...,5888,101,5787,0.011059,1.0,0,0,0,1,2
11,pydantic,0,39,107045,1,1,0,0,0,0,...,3314,8,3305,0.061422,1.0,0,0,0,1,2
3,django,0,96,2017,1,1,0,0,0,0,...,5857,10,5846,0.073897,1.0,0,0,0,1,2


In [6]:

from sklearn.model_selection import train_test_split

stratify_values = data[target_column] if data[target_column].nunique() > 1 and data[target_column].value_counts().min() >= 3 else None

train_df, temp_df = train_test_split(
    data,
    train_size=0.80,
    random_state=42,
    stratify=stratify_values,
)

temp_stratify = temp_df[target_column] if temp_df[target_column].nunique() > 1 and temp_df[target_column].value_counts().min() >= 2 else None
val_df, batch_df = train_test_split(
    temp_df,
    train_size=0.50,
    random_state=42,
    stratify=temp_stratify,
)

# SageMaker built-in XGBoost expects the target as the first column for CSV training.
data_train = train_df[[target_column] + model_feature_columns].copy()
data_val = val_df[[target_column] + model_feature_columns].copy()

# Batch input keeps the identifier first, then the exact numeric model features.
data_batch = batch_df[[id_column] + model_feature_columns].copy()
data_batch_noID = data_batch.drop([id_column], axis=1)

positive_count = int(data_train[target_column].sum())
negative_count = int(len(data_train) - positive_count)
scale_pos_weight = negative_count / max(positive_count, 1)

print("Train shape:", data_train.shape)
print("Validation shape:", data_val.shape)
print("Batch shape:", data_batch.shape)
print("scale_pos_weight:", scale_pos_weight)


Train shape: (16, 22)
Validation shape: (2, 22)
Batch shape: (2, 22)
scale_pos_weight: 16.0


Let's upload those data sets in S3

In [7]:
train_file = "train_data.csv"
data_train.to_csv(train_file, index=False, header=False)
sess.upload_data(train_file, key_prefix="{}/train".format(prefix))

validation_file = "validation_data.csv"
data_val.to_csv(validation_file, index=False, header=False)
sess.upload_data(validation_file, key_prefix="{}/validation".format(prefix))

batch_file = "batch_data.csv"
data_batch.to_csv(batch_file, index=False, header=False)
sess.upload_data(batch_file, key_prefix="{}/batch".format(prefix))

batch_file_noID = "batch_data_noID.csv"
data_batch_noID.to_csv(batch_file_noID, index=False, header=False)
sess.upload_data(batch_file_noID, key_prefix="{}/batch".format(prefix))

's3://sagemaker-us-east-1-670238798194/packageguard-ai-risk-scoring-xgboost-highlevel/batch/batch_data_noID.csv'

---

## Training job and model creation

In [8]:
%%time
from time import gmtime, strftime

job_name = "packageguard-xgb-" + strftime("%Y-%m-%d-%H-%M-%S", gmtime())
output_location = "s3://{}/{}/output/{}".format(bucket, prefix, job_name)

xgboost_image_by_region = {
    "us-east-1": "683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1",
    "us-east-2": "257758044811.dkr.ecr.us-east-2.amazonaws.com/sagemaker-xgboost:1.7-1",
    "us-west-1": "746614075791.dkr.ecr.us-west-1.amazonaws.com/sagemaker-xgboost:1.7-1",
    "us-west-2": "433757028032.dkr.ecr.us-west-2.amazonaws.com/sagemaker-xgboost:1.7-1",
}

if region not in xgboost_image_by_region:
    raise ValueError(f"Unsupported region for this notebook image map: {region}")

image = xgboost_image_by_region[region]

train_s3_uri = "s3://{}/{}/train".format(bucket, prefix)
validation_s3_uri = "s3://{}/{}/validation".format(bucket, prefix)

print("Training job:", job_name, flush=True)
print("Region:", region, flush=True)
print("XGBoost image:", image, flush=True)
print("Output location:", output_location, flush=True)
print("Train data:", train_s3_uri, flush=True)
print("Validation data:", validation_s3_uri, flush=True)

sm_client.create_training_job(
    TrainingJobName=job_name,
    AlgorithmSpecification={
        "TrainingImage": image,
        "TrainingInputMode": "File",
    },
    RoleArn=role,
    InputDataConfig=[
        {
            "ChannelName": "train",
            "DataSource": {
                "S3DataSource": {
                    "S3DataType": "S3Prefix",
                    "S3Uri": train_s3_uri,
                    "S3DataDistributionType": "FullyReplicated",
                }
            },
            "ContentType": "text/csv",
            "CompressionType": "None",
        },
        {
            "ChannelName": "validation",
            "DataSource": {
                "S3DataSource": {
                    "S3DataType": "S3Prefix",
                    "S3Uri": validation_s3_uri,
                    "S3DataDistributionType": "FullyReplicated",
                }
            },
            "ContentType": "text/csv",
            "CompressionType": "None",
        },
    ],
    OutputDataConfig={
        "S3OutputPath": output_location
    },
    ResourceConfig={
        "InstanceType": "ml.m5.xlarge",
        "InstanceCount": 1,
        "VolumeSizeInGB": 50,
    },
    StoppingCondition={
        "MaxRuntimeInSeconds": 3600
    },
    HyperParameters={
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "max_depth": "4",
        "eta": "0.1",
        "gamma": "2",
        "min_child_weight": "3",
        "subsample": "0.8",
        "colsample_bytree": "0.8",
        "scale_pos_weight": str(scale_pos_weight),
        "verbosity": "1",
        "num_round": "150",
    },
)

print("Started training job:", job_name, flush=True)

sm_client.get_waiter("training_job_completed_or_stopped").wait(
    TrainingJobName=job_name
)

training_info = sm_client.describe_training_job(
    TrainingJobName=job_name
)

print("Training status:", training_info["TrainingJobStatus"], flush=True)

if training_info["TrainingJobStatus"] != "Completed":
    raise RuntimeError(
        "Training job did not complete successfully. Status: {}".format(
            training_info["TrainingJobStatus"]
        )
    )

model_data = training_info["ModelArtifacts"]["S3ModelArtifacts"]

print("Model artifact:", model_data, flush=True)


Training job: packageguard-xgb-2026-05-31-06-17-34


Region: us-east-1


XGBoost image: 683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1


Output location: s3://sagemaker-us-east-1-670238798194/packageguard-ai-risk-scoring-xgboost-highlevel/output/packageguard-xgb-2026-05-31-06-17-34


Train data: s3://sagemaker-us-east-1-670238798194/packageguard-ai-risk-scoring-xgboost-highlevel/train


Validation data: s3://sagemaker-us-east-1-670238798194/packageguard-ai-risk-scoring-xgboost-highlevel/validation


Started training job: packageguard-xgb-2026-05-31-06-17-34


Training status: Completed


Model artifact: s3://sagemaker-us-east-1-670238798194/packageguard-ai-risk-scoring-xgboost-highlevel/output/packageguard-xgb-2026-05-31-06-17-34/packageguard-xgb-2026-05-31-06-17-34/output/model.tar.gz


CPU times: user 74.5 ms, sys: 29.3 ms, total: 104 ms
Wall time: 4min


## Batch Transform

## 1. Create a transform job with the default configurations


In [9]:
%%time
from time import gmtime, strftime

transform_job_name = "packageguard-transform-" + strftime("%Y-%m-%d-%H-%M-%S", gmtime())
model_name = "packageguard-model-" + strftime("%Y-%m-%d-%H-%M-%S", gmtime())

create_model_response = sm_client.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={
        "Image": image,
        "ModelDataUrl": model_data,
    },
)

input_location = "s3://{}/{}/batch/{}".format(
    bucket, prefix, batch_file_noID
)

batch_output_location = "s3://{}/{}/batch-output/{}".format(
    bucket, prefix, transform_job_name
)

print("Created SageMaker model:", model_name, flush=True)
print("Created SageMaker Model ARN:", create_model_response["ModelArn"], flush=True)
print("Transform job:", transform_job_name, flush=True)
print("Input:", input_location, flush=True)
print("Output:", batch_output_location, flush=True)

sm_client.create_transform_job(
    TransformJobName=transform_job_name,
    ModelName=model_name,
    TransformInput={
        "DataSource": {
            "S3DataSource": {
                "S3DataType": "S3Prefix",
                "S3Uri": input_location,
            }
        },
        "ContentType": "text/csv",
        "SplitType": "Line",
        "CompressionType": "None",
    },
    TransformOutput={
        "S3OutputPath": batch_output_location,
        "Accept": "text/csv",
        "AssembleWith": "Line",
    },
    TransformResources={
        "InstanceType": "ml.m5.xlarge",
        "InstanceCount": 1,
    },
)

sm_client.get_waiter("transform_job_completed_or_stopped").wait(
    TransformJobName=transform_job_name
)

transform_info = sm_client.describe_transform_job(
    TransformJobName=transform_job_name
)

print("Transform status:", transform_info["TransformJobStatus"], flush=True)
print("Batch output location:", batch_output_location, flush=True)


Created SageMaker model: packageguard-model-2026-05-31-06-21-41


Created SageMaker Model ARN: arn:aws:sagemaker:us-east-1:670238798194:model/packageguard-model-2026-05-31-06-21-41


Transform job: packageguard-transform-2026-05-31-06-21-41


Input: s3://sagemaker-us-east-1-670238798194/packageguard-ai-risk-scoring-xgboost-highlevel/batch/batch_data_noID.csv


Output: s3://sagemaker-us-east-1-670238798194/packageguard-ai-risk-scoring-xgboost-highlevel/batch-output/packageguard-transform-2026-05-31-06-21-41


Transform status: Completed


Batch output location: s3://sagemaker-us-east-1-670238798194/packageguard-ai-risk-scoring-xgboost-highlevel/batch-output/packageguard-transform-2026-05-31-06-21-41


CPU times: user 84.5 ms, sys: 39.4 ms, total: 124 ms
Wall time: 5min 1s


# Inspect the output of the Batch Transform job in S3.


In [10]:
import re


def get_csv_output_from_s3(s3uri, batch_file):
    file_name = "{}.out".format(batch_file)
    match = re.match("s3://([^/]+)/(.*)", "{}/{}".format(s3uri, file_name))
    output_bucket, output_prefix = match.group(1), match.group(2)
    s3.download_file(output_bucket, output_prefix, file_name)
    return pd.read_csv(file_name, sep=",", header=None)

In [11]:
output_df = get_csv_output_from_s3(
    batch_output_location,
    batch_file_noID
)

output_df.head(8)


,0
0,0.247896
1,0.247896


## 2. Join the input and the prediction results
Now, let's associate the prediction results with their corresponding input records. We can also use the __input_filter__ to exclude the package identifier column easily and there is no need to have a separate file in S3.

* Set __input_filter__ to `$[1:]`: exclude column 0 (`package_name`) before processing inference and keep all model features.
* Set __join_source__ to `Input`: join the original input row with the inference result.
* Leave __output_filter__ as default (`$`): save the joined input and risk score as output.


In [12]:
%%time
from time import gmtime, strftime

transform_job_name_join = "packageguard-transform-join-" + strftime(
    "%Y-%m-%d-%H-%M-%S", gmtime()
)

input_location_join = "s3://{}/{}/batch/{}".format(
    bucket, prefix, batch_file
)

batch_output_location_join = "s3://{}/{}/batch-output/{}".format(
    bucket, prefix, transform_job_name_join
)

print("Transform job:", transform_job_name_join, flush=True)
print("Input:", input_location_join, flush=True)
print("Output:", batch_output_location_join, flush=True)

sm_client.create_transform_job(
    TransformJobName=transform_job_name_join,
    ModelName=model_name,
    TransformInput={
        "DataSource": {
            "S3DataSource": {
                "S3DataType": "S3Prefix",
                "S3Uri": input_location_join,
            }
        },
        "ContentType": "text/csv",
        "SplitType": "Line",
        "CompressionType": "None",
    },
    TransformOutput={
        "S3OutputPath": batch_output_location_join,
        "Accept": "text/csv",
        "AssembleWith": "Line",
    },
    TransformResources={
        "InstanceType": "ml.m5.xlarge",
        "InstanceCount": 1,
    },
    DataProcessing={
        # Remove first column before inference.
        # For PackageGuard, first column is the package_name identifier.
        "InputFilter": "$[1:]",

        # Join original input row with prediction result.
        "JoinSource": "Input",
    },
)

sm_client.get_waiter("transform_job_completed_or_stopped").wait(
    TransformJobName=transform_job_name_join
)

transform_info_join = sm_client.describe_transform_job(
    TransformJobName=transform_job_name_join
)

print("Transform status:", transform_info_join["TransformJobStatus"], flush=True)
print("Batch output location:", batch_output_location_join, flush=True)


Transform job: packageguard-transform-join-2026-05-31-06-27-02


Input: s3://sagemaker-us-east-1-670238798194/packageguard-ai-risk-scoring-xgboost-highlevel/batch/batch_data.csv


Output: s3://sagemaker-us-east-1-670238798194/packageguard-ai-risk-scoring-xgboost-highlevel/batch-output/packageguard-transform-join-2026-05-31-06-27-02


Transform status: Completed


Batch output location: s3://sagemaker-us-east-1-670238798194/packageguard-ai-risk-scoring-xgboost-highlevel/batch-output/packageguard-transform-join-2026-05-31-06-27-02


CPU times: user 105 ms, sys: 27.8 ms, total: 132 ms
Wall time: 6min


# Inspect the output of the Batch Transform job in S3.


In [13]:
output_df = get_csv_output_from_s3(
    batch_output_location_join,
    batch_file
)

output_df.head(8)


,0,1,2,3,4,5,6,7,8,9,...,13,14,15,16,17,18,19,20,21,22
0,tensorflow,69,854,1,1,1,0,1,1,34,...,85,3363,0.041035,1.0,0,0,1,0,5,0.247896
1,boto3,22,5368,1,1,1,0,1,1,4,...,1,4216,0.484345,1.0,0,0,0,0,5,0.247896


#### 3. Update the output filter to keep only package name and prediction result
Let's change __output_filter__ to `$[0,-1]`, indicating that when presenting the output, we only want to keep column 0 (`package_name`) and the last column, which is the model's suspiciousness/risk score.


In [14]:
%%time
from time import gmtime, strftime

transform_job_name_id_score = "packageguard-transform-id-score-" + strftime(
    "%Y-%m-%d-%H-%M-%S", gmtime()
)

input_location_id_score = "s3://{}/{}/batch/{}".format(
    bucket, prefix, batch_file
)

batch_output_location_id_score = "s3://{}/{}/batch-output/{}".format(
    bucket, prefix, transform_job_name_id_score
)

print("Transform job:", transform_job_name_id_score, flush=True)
print("Input:", input_location_id_score, flush=True)
print("Output:", batch_output_location_id_score, flush=True)

sm_client.create_transform_job(
    TransformJobName=transform_job_name_id_score,
    ModelName=model_name,
    TransformInput={
        "DataSource": {
            "S3DataSource": {
                "S3DataType": "S3Prefix",
                "S3Uri": input_location_id_score,
            }
        },
        "ContentType": "text/csv",
        "SplitType": "Line",
        "CompressionType": "None",
    },
    TransformOutput={
        "S3OutputPath": batch_output_location_id_score,
        "Accept": "text/csv",
        "AssembleWith": "Line",
    },
    TransformResources={
        "InstanceType": "ml.m5.xlarge",
        "InstanceCount": 1,
    },
    DataProcessing={
        # Send only feature columns to the model.
        # Column 0 is package_name and is excluded before prediction.
        "InputFilter": "$[1:]",

        # Join original input with model prediction.
        "JoinSource": "Input",

        # Keep only package_name and prediction.
        # Output becomes: package_name,risk_score
        "OutputFilter": "$[0,-1]",
    },
)

sm_client.get_waiter("transform_job_completed_or_stopped").wait(
    TransformJobName=transform_job_name_id_score
)

transform_info_id_score = sm_client.describe_transform_job(
    TransformJobName=transform_job_name_id_score
)

print("Transform status:", transform_info_id_score["TransformJobStatus"], flush=True)
print("Batch output location:", batch_output_location_id_score, flush=True)


Transform job: packageguard-transform-id-score-2026-05-31-06-33-15


Input: s3://sagemaker-us-east-1-670238798194/packageguard-ai-risk-scoring-xgboost-highlevel/batch/batch_data.csv


Output: s3://sagemaker-us-east-1-670238798194/packageguard-ai-risk-scoring-xgboost-highlevel/batch-output/packageguard-transform-id-score-2026-05-31-06-33-15


Transform status: Completed


Batch output location: s3://sagemaker-us-east-1-670238798194/packageguard-ai-risk-scoring-xgboost-highlevel/batch-output/packageguard-transform-id-score-2026-05-31-06-33-15


CPU times: user 92.4 ms, sys: 38.9 ms, total: 131 ms
Wall time: 6min


# inspect the output of the Batch Transform job in S3


In [15]:
output_df = get_csv_output_from_s3(
    batch_output_location_id_score,
    batch_file
)

output_df.columns = ["package_name", "risk_score"]

output_df.head(8)


,package_name,risk_score
0,tensorflow,0.247896
1,boto3,0.247896


# Part 1: Set Up Model Group

In [16]:
from botocore.exceptions import ClientError
import json
from pprint import pprint

model_package_group_name = "packageguard-pypi-risk-scoring"
model_package_group_description = (
    "Versioned PackageGuard models that score PyPI package metadata for suspiciousness and human security review."
)

try:
    create_model_package_group_response = sm_client.create_model_package_group(
        ModelPackageGroupName=model_package_group_name,
        ModelPackageGroupDescription=model_package_group_description,
    )
    print("Created Model Package Group:")
    print(create_model_package_group_response["ModelPackageGroupArn"])

except ClientError as error:
    if error.response["Error"]["Code"] == "ValidationException" and "already exists" in str(error):
        print(f"Model Package Group already exists: {model_package_group_name}")
    else:
        raise


Model Package Group already exists: packageguard-pypi-risk-scoring


In [17]:
describe_model_package_group_response = sm_client.describe_model_package_group(
    ModelPackageGroupName=model_package_group_name
)

pprint(describe_model_package_group_response)


{'CreatedBy': {'DomainId': 'd-2wdrhdmzc9kc',
               'IamIdentity': {'Arn': 'arn:aws:sts::670238798194:assumed-role/LabRole/SageMaker',
                               'PrincipalId': 'AROAZYDKYOFZBE2RZEAKW:SageMaker'},
               'UserProfileArn': 'arn:aws:sagemaker:us-east-1:670238798194:user-profile/d-2wdrhdmzc9kc/default-1778549832679',
               'UserProfileName': 'default-1778549832679'},
 'CreationTime': datetime.datetime(2026, 5, 31, 5, 29, 6, 780000, tzinfo=tzlocal()),
 'ModelPackageGroupArn': 'arn:aws:sagemaker:us-east-1:670238798194:model-package-group/packageguard-pypi-risk-scoring',
 'ModelPackageGroupDescription': 'Versioned PackageGuard models that score '
                                 'PyPI package metadata for suspiciousness and '
                                 'human security review.',
 'ModelPackageGroupName': 'packageguard-pypi-risk-scoring',
 'ModelPackageGroupStatus': 'Completed',
 'ResponseMetadata': {'HTTPHeaders': {'cache-control': 'no-cache,

# Part 2: Set Up Model Package

In [18]:
model_package_description = (
    "XGBoost risk scoring model for PyPI packages. "
    "Outputs suspiciousness probability for human review."
)

model_approval_status = "PendingManualApproval"

# SageMaker CustomerMetadataProperties is strict: each value must be short
# and must avoid unsupported punctuation. Keep detailed text in the Model Card.
safe_feature_summary = "_".join(model_feature_columns[:5])[:250]

customer_metadata_properties = {
    "project": "PackageGuardAI",
    "ecosystem": "PyPI",
    "model_type": "XGBoostBinaryClassifier",
    "prediction_output": "risk_score_probability",
    "human_review_required": "true",
    "training_job_name": job_name,
    "data_source": "OpenSSF_DataDog_PyPI_JSON_API",
    "feature_summary": safe_feature_summary,
}

create_model_package_response = sm_client.create_model_package(
    ModelPackageGroupName=model_package_group_name,
    ModelPackageDescription=model_package_description,
    InferenceSpecification={
        "Containers": [
            {
                "Image": image,
                "ModelDataUrl": model_data,
            }
        ],
        "SupportedContentTypes": ["text/csv"],
        "SupportedResponseMIMETypes": ["text/csv"],
    },
    ModelApprovalStatus=model_approval_status,
    CustomerMetadataProperties=customer_metadata_properties,
)

model_package_arn = create_model_package_response["ModelPackageArn"]

print("Created Model Package ARN:")
print(model_package_arn)


Created Model Package ARN:
arn:aws:sagemaker:us-east-1:670238798194:model-package/packageguard-pypi-risk-scoring/2


In [19]:
describe_model_package_response = sm_client.describe_model_package(
    ModelPackageName=model_package_arn
)

pprint(describe_model_package_response)


{'CertifyForMarketplace': False,
 'CreatedBy': {'DomainId': 'd-2wdrhdmzc9kc',
               'IamIdentity': {'Arn': 'arn:aws:sts::670238798194:assumed-role/LabRole/SageMaker',
                               'PrincipalId': 'AROAZYDKYOFZBE2RZEAKW:SageMaker'},
               'UserProfileArn': 'arn:aws:sagemaker:us-east-1:670238798194:user-profile/d-2wdrhdmzc9kc/default-1778549832679',
               'UserProfileName': 'default-1778549832679'},
 'CreationTime': datetime.datetime(2026, 5, 31, 6, 42, 56, 40000, tzinfo=tzlocal()),
 'CustomerMetadataProperties': {'data_source': 'OpenSSF_DataDog_PyPI_JSON_API',
                                'ecosystem': 'PyPI',
                                'feature_summary': 'summary_length_description_length_has_summary_has_description_has_author',
                                'human_review_required': 'true',
                                'model_type': 'XGBoostBinaryClassifier',
                                'prediction_output': 'risk_score_probabi

# Part 3: Write the Model Card

In [20]:

import json
from time import gmtime, strftime

model_card_name = "packageguard-pypi-risk-scoring-card-" + strftime("%Y-%m-%d-%H-%M-%S", gmtime())

model_card_content = {
    "model_overview": {
        "model_name": model_name,
        "model_id": model_package_arn,
        "model_description": (
            "PackageGuard PyPI risk-scoring model that assigns suspiciousness probabilities "
            "to packages using engineered package metadata and ecosystem signals."
        ),
        "model_creator": "Student project team",
        "model_owner": "Student project team",
        "problem_type": "Binary classification / risk scoring",
        "algorithm_type": "XGBoost binary:logistic",
        "model_artifact": [model_data],
    },
    "intended_uses": {
        "purpose_of_model": (
            "Support software supply-chain security triage by prioritizing PyPI packages "
            "that may require human review."
        ),
        "intended_uses": (
            "Early-warning risk scoring, analyst triage, package review prioritization, "
            "and model governance demonstration in SageMaker."
        ),
        "risk_rating": "Medium",
        "explanations_for_risk_rating": (
            "False positives may flag benign packages and false negatives may miss suspicious packages. "
            "Human review is required before enforcement. This model must not be used as a final malware judgment system."
        ),
    },
    "business_details": {
        "business_problem": (
            "Open-source package ecosystems can contain malicious or suspicious packages. "
            "Security teams need a lightweight early-warning score to prioritize review."
        ),
        "business_stakeholders": (
            "Security analysts, ML engineers, MLOps team, package governance team"
        ),
        "line_of_business": "Software supply-chain security",
    },
    "training_details": {
        "objective_function": {
            "function": "binary:logistic",
            "notes": "Predicts a probability-like risk score between 0 and 1.",
        },
        "training_observations": (
            "Training used engineered PyPI metadata features. The target label indicates whether "
            "a package appeared in curated malicious package sources. XGBoost hyperparameters included "
            "objective=binary:logistic, eval_metric=auc, max_depth=4, eta=0.1, gamma=2, "
            "min_child_weight=3, subsample=0.8, colsample_bytree=0.8, and num_round=150."
        ),
        "training_job_details": {
            "training_arn": training_info["TrainingJobArn"],
            "training_datasets": [
                f"s3://{bucket}/{prefix}/train",
                f"s3://{bucket}/{prefix}/validation",
            ],
            "training_environment": {
                "container_image": [image]
            },
            "training_metrics": [
                {
                    "name": "validation_auc",
                    "value": 0.0,
                    "notes": "AUC was configured as the XGBoost validation metric. See training logs for final value.",
                }
            ],
            "user_provided_training_metrics": [
                {
                    "name": "num_round",
                    "value": 150,
                    "notes": "Number of boosting rounds.",
                },
                {
                    "name": "max_depth",
                    "value": 4,
                    "notes": "Maximum tree depth.",
                },
                {
                    "name": "eta",
                    "value": 0.1,
                    "notes": "Learning rate.",
                },
            ],
        },
    },
    "evaluation_details": [
        {
            "name": "Validation during XGBoost training",
            "evaluation_observation": (
                "The model was trained with a separate validation split and AUC as the evaluation metric. "
                "Batch Transform outputs should be reviewed as risk scores rather than hard malware labels."
            ),
            "datasets": ["validation_data.csv", "batch_data.csv"],
            "metric_groups": [
                {
                    "name": "Training validation metric",
                    "metric_data": [
                        {
                            "name": "AUC",
                            "type": "number",
                            "value": 0.0,
                            "notes": "Placeholder. Review CloudWatch/SageMaker training logs for the final validation AUC.",
                        }
                    ],
                }
            ],
        }
    ],
    "additional_information": {
        "ethical_considerations": (
            "The model may encode bias from public malicious-package lists and metadata availability. "
            "Use as decision support only."
        ),
        "caveats_and_recommendations": (
            "A suspicious score should trigger human review. Do not use this model for automatic blocking "
            "or final malware judgment without analyst validation."
        ),
        "custom_details": {
            "input_features": ", ".join(model_feature_columns[:20]),
            "identifier_column": id_column,
            "target_column": target_column,
            "score_threshold_note": (
                "0.50 is only a demo review threshold, not a production enforcement threshold."
            ),
        },
    },
}

create_model_card_response = sm_client.create_model_card(
    ModelCardName=model_card_name,
    Content=json.dumps(model_card_content),
    ModelCardStatus="Draft",
)

print("Created Model Card ARN:")
print(create_model_card_response["ModelCardArn"])
print("Model Card Name:")
print(model_card_name)


Created Model Card ARN:
arn:aws:sagemaker:us-east-1:670238798194:model-card/packageguard-pypi-risk-scoring-card-2026-05-31-06-43-07
Model Card Name:
packageguard-pypi-risk-scoring-card-2026-05-31-06-43-07


In [21]:
# Model Card Response
describe_model_card_response = sm_client.describe_model_card(
    ModelCardName=model_card_name
)

pprint(describe_model_card_response)


{'Content': '{"model_overview": {"model_name": '
            '"packageguard-model-2026-05-31-06-21-41", "model_id": '
            '"arn:aws:sagemaker:us-east-1:670238798194:model-package/packageguard-pypi-risk-scoring/2", '
            '"model_description": "PackageGuard PyPI risk-scoring model that '
            'assigns suspiciousness probabilities to packages using engineered '
            'package metadata and ecosystem signals.", "model_creator": '
            '"Student project team", "model_owner": "Student project team", '
            '"problem_type": "Binary classification / risk scoring", '
            '"algorithm_type": "XGBoost binary:logistic", "model_artifact": '
            '["s3://sagemaker-us-east-1-670238798194/packageguard-ai-risk-scoring-xgboost-highlevel/output/packageguard-xgb-2026-05-31-06-17-34/packageguard-xgb-2026-05-31-06-17-34/output/model.tar.gz"]}, '
            '"intended_uses": {"purpose_of_model": "Support software '
            'supply-chain security tr